# Plant Disease Detector — Training on PlantVillage

**Avishkar 2026** · CNN leaf-disease classifier with Marathi / Hindi advisory

This notebook trains the model that powers the farmer app. Run it on **Google Colab with a GPU**:

> `Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU` → Save

**What it produces:** `plant_disease_model.pt` — a single self-describing checkpoint holding the
weights, the class list, and the validation metrics. Download it and drop it into
`backend/models/` in the project folder. That is the only file the app needs.

**Expected result:** ~99% validation accuracy on the PlantVillage split, ~25 minutes on a free T4.

---
### Method summary (for your report / viva)

| Choice | Value | Why |
|---|---|---|
| Architecture | MobileNetV3-Large | 5.4M params, ~22 MB — runs on a CPU laptop in <1s per image |
| Training | Transfer learning from ImageNet | 54k images is not enough to learn low-level filters from scratch |
| Schedule | 2 warm-up epochs (head only), then full fine-tune | Prevents the random head from wrecking pretrained features |
| Loss | Cross-entropy with label smoothing 0.1 | Stops the model being over-confident, which matters for our 60% cut-off |
| Augmentation | Crop, flip, rotate, colour jitter, random erasing | PlantVillage is shot on plain backgrounds; a phone photo is not |
| Split | 80 / 20 stratified | Every one of the 38 classes appears in both splits |


## 1. Check the GPU

If this prints *No GPU*, change the runtime type before going further.

In [ ]:
import torch, subprocess

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

print("torch", torch.__version__)

## 2. Get the PlantVillage dataset

**Option A — Kaggle (faster, ~700 MB).** Get `kaggle.json` from
[kaggle.com](https://www.kaggle.com) → *Settings* → *API* → *Create New Token*, then run the
cell and upload it.

**Option B — GitHub mirror.** No account needed, but clones ~2 GB. Use it if Kaggle fails.

Both give the same 38-class colour split: 54,305 leaf images across 14 crops.

In [ ]:
# --- Option A: Kaggle ---
import os, pathlib

from google.colab import files
print("Upload your kaggle.json:")
files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content --unzip --quiet

DATA_DIR = "/content/plantvillage dataset/color"
print("Exists:", os.path.isdir(DATA_DIR))

In [ ]:
# --- Option B: GitHub mirror (run ONLY if Option A failed) ---
# !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/pv
# DATA_DIR = "/content/pv/raw/color"
# print("Exists:", os.path.isdir(DATA_DIR))

In [ ]:
import os
from collections import Counter

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes}

print(f"{len(classes)} classes, {sum(counts.values()):,} images\n")
for c in classes:
    print(f"{counts[c]:>6,}  {c}")

# Sanity check against the advisory file the app ships with.
assert len(classes) == 38, f"Expected 38 classes, found {len(classes)} — check DATA_DIR"

## 3. Model and transforms

These definitions mirror `backend/app/ml/model.py`. The checkpoint stores the `arch` string, so the
server rebuilds the exact same network before loading the weights.

In [ ]:
import torch.nn as nn
from torchvision import models, transforms

ARCH = "mobilenet_v3_large"
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


def build_model(num_classes, arch=ARCH, pretrained=True):
    if arch == "mobilenet_v3_large":
        w = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        m = models.mobilenet_v3_large(weights=w)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif arch == "efficientnet_b0":
        w = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.efficientnet_b0(weights=w)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif arch == "resnet18":
        w = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.resnet18(weights=w)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(arch)
    return m


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("transforms ready")

## 4. Stratified 80 / 20 split

PlantVillage ships as one folder per class with no train/val split, and the classes are
imbalanced (from 152 to 5,507 images). A plain random split can leave a rare class badly
represented in validation, so we split **within each class**.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

SEED = 42
BATCH_SIZE = 64
VAL_FRACTION = 0.2

torch.manual_seed(SEED)
np.random.seed(SEED)

base = ImageFolder(DATA_DIR)          # loads paths + targets only
class_names = base.classes
targets = np.array(base.targets)

train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=VAL_FRACTION,
    stratify=targets,        # <- the important bit
    random_state=SEED,
)

# Two dataset objects over the same files so each split gets its own transform.
train_ds = Subset(ImageFolder(DATA_DIR, transform=train_tf), train_idx)
val_ds   = Subset(ImageFolder(DATA_DIR, transform=eval_tf),  val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print(f"train {len(train_ds):,}   val {len(val_ds):,}   classes {len(class_names)}")

## 5. Look at the data

Always eyeball a batch before training — a wrong `DATA_DIR` shows up here immediately.

In [ ]:
import matplotlib.pyplot as plt

def denorm(t):
    return (t * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)).clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, img, lab in zip(axes.flat, imgs, labels):
    ax.imshow(denorm(img).permute(1, 2, 0))
    name = class_names[lab].replace("___", "\n").replace("_", " ")
    ax.set_title(name, fontsize=7)
    ax.axis("off")
plt.suptitle("Augmented training samples", fontsize=13)
plt.tight_layout(); plt.show()

## 6. Train

Two stages:

1. **Warm-up (2 epochs)** — backbone frozen, only the new classifier learns. Without this the
   randomly-initialised head sends large gradients back and damages the pretrained filters.
2. **Fine-tune (8 epochs)** — everything unfrozen at a 10× lower learning rate, with a cosine
   schedule and mixed precision.

Roughly 25 minutes on a free T4.

In [ ]:
import time, copy
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(len(class_names)).to(device)

WARMUP_EPOCHS = 2
FINETUNE_EPOCHS = 8
EPOCHS = WARMUP_EPOCHS + FINETUNE_EPOCHS

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")


def set_backbone_trainable(flag):
    for name, p in model.named_parameters():
        if not name.startswith("classifier"):
            p.requires_grad = flag


def run_epoch(loader, train):
    model.train(train)
    total_loss = correct = seen = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        seen += y.size(0)
    return total_loss / seen, correct / seen


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc, best_state = 0.0, None
started = time.time()

for epoch in range(1, EPOCHS + 1):
    if epoch == 1:
        set_backbone_trainable(False)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
        scheduler = None
        print(f"-- warm-up: classifier only, lr 1e-3")
    elif epoch == WARMUP_EPOCHS + 1:
        set_backbone_trainable(True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)
        print(f"-- fine-tune: all layers, lr 1e-4 with cosine decay")

    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    if scheduler: scheduler.step()

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)

    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        best_state = copy.deepcopy(model.state_dict())
        flag = "  <- best"

    print(f"epoch {epoch:2d}/{EPOCHS}  "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f}  |  "
          f"val loss {va_loss:.4f} acc {va_acc:.4f}  "
          f"({time.time()-t0:.0f}s){flag}")

print(f"\nBest validation accuracy: {best_acc:.4f}  "
      f"(total {(time.time()-started)/60:.1f} min)")
model.load_state_dict(best_state)

## 7. Training curves

Save these figures — they belong on your poster.

In [ ]:
epochs = range(1, len(history["train_acc"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))

a1.plot(epochs, history["train_acc"], "o-", label="train")
a1.plot(epochs, history["val_acc"], "s-", label="validation")
a1.axvline(WARMUP_EPOCHS + 0.5, ls="--", c="grey", lw=1)
a1.set_title("Accuracy"); a1.set_xlabel("epoch"); a1.legend(); a1.grid(alpha=.3)

a2.plot(epochs, history["train_loss"], "o-", label="train")
a2.plot(epochs, history["val_loss"], "s-", label="validation")
a2.axvline(WARMUP_EPOCHS + 0.5, ls="--", c="grey", lw=1)
a2.set_title("Loss"); a2.set_xlabel("epoch"); a2.legend(); a2.grid(alpha=.3)

plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 8. Per-class evaluation

Overall accuracy hides the classes the model is actually bad at. The report below is what a
judge will ask about — know your two or three worst classes and why they are confused
(usually Tomato Early blight vs Target Spot, which genuinely look alike).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
y_true, y_pred = [], []
with torch.inference_mode():
    for x, y in val_loader:
        out = model(x.to(device))
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_true.extend(y.tolist())

short = [c.replace("___", " / ").replace("_", " ")[:34] for c in class_names]
print(classification_report(y_true, y_pred, target_names=short, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(short))); ax.set_xticklabels(short, rotation=90, fontsize=7)
ax.set_yticks(range(len(short))); ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Normalised confusion matrix (validation)")
fig.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

# The most-confused pairs, worth naming in the viva.
import numpy as np
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
print("\nTop confusions:")
for i, j in zip(*np.unravel_index(np.argsort(cm_off, axis=None)[::-1][:8], cm_off.shape)):
    print(f"  {cm_off[i, j]*100:5.1f}%  {short[i]}  ->  {short[j]}")

## 9. Save the checkpoint

One file, self-describing: weights + class list + metrics. `backend/app/services/inference.py` reads all
three, so you never have to keep a separate labels file in sync.

In [ ]:
metrics = {
    "val_accuracy": best_acc,
    "epochs": EPOCHS,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "batch_size": BATCH_SIZE,
    "history": history,
}

torch.save({
    "arch": ARCH,
    "img_size": IMG_SIZE,
    "class_names": class_names,
    "state_dict": model.state_dict(),
    "metrics": metrics,
}, "plant_disease_model.pt")

size_mb = os.path.getsize("plant_disease_model.pt") / 1e6
print(f"saved plant_disease_model.pt  ({size_mb:.1f} MB, val acc {best_acc:.4f})")

## 10. Verify the checkpoint loads cleanly

This reloads the file from scratch on the CPU, exactly the way the server will. If it passes
here it will work on your laptop.

In [ ]:
from PIL import Image

ckpt = torch.load("plant_disease_model.pt", map_location="cpu", weights_only=False)
check = build_model(len(ckpt["class_names"]), arch=ckpt["arch"], pretrained=False)
check.load_state_dict(ckpt["state_dict"])
check.eval()

# Predict one real validation image end to end.
path, label = base.samples[val_idx[0]]
img = Image.open(path).convert("RGB")
with torch.inference_mode():
    probs = torch.softmax(check(eval_tf(img).unsqueeze(0)), 1)[0]

top = torch.topk(probs, 3)
print("file :", os.path.basename(path))
print("truth:", class_names[label])
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {p*100:6.2f}%  {class_names[i]}")

## 11. Download

Save the file, then put it at:

```
Avishkar - 2026/backend/models/plant_disease_model.pt
```

Restart the server and the app is live.

In [ ]:
from google.colab import files

files.download("plant_disease_model.pt")
files.download("training_curves.png")
files.download("confusion_matrix.png")